In [1]:
import re
import numpy as np
import pandas as pd
import torch
 
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [2]:
# =============================================================
# SECTION 1 — LOAD SOLD DATASET
# =============================================================
 
sold = load_dataset("sinhala-nlp/SOLD")
 
train_df = sold["train"].to_pandas()
test_df  = sold["test"].to_pandas()
 
label_map = {"NOT": 0, "OFF": 1}
train_df["label_num"] = train_df["label"].map(label_map)
test_df["label_num"]  = test_df["label"].map(label_map)
 
print("SOLD loaded:")
print(f"  Train: {train_df.shape}, Test: {test_df.shape}")
print(train_df["label"].value_counts())

README.md: 0.00B [00:00, ?B/s]

SOLD_train.tsv: 0.00B [00:00, ?B/s]

SOLD_test.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/7500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

SOLD loaded:
  Train: (7500, 6), Test: (2500, 6)
label
NOT    4324
OFF    3176
Name: count, dtype: int64


In [3]:
train_df

,post_id,text,tokens,rationales,label,label_num
0,726758237668659201,@USER @USER පට්ට පට පට...,@USER @USER පට්ට පට පට . . .,[],NOT,0
1,915618589855617026,පරණ කෑල්ල අද වෙනකම් හිටියනම් අදට අවුරුදු 4යි. ...,පරණ කෑල්ල අද වෙනකම් හිටියනම් අදට අවුරුදු 4යි ....,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",OFF,1
2,925001070430040065,යාළුවා කියලා හිතන් සර් ගේ ඔලුවට රෙද්ද දාලා නෙල...,යාළුවා කියලා හිතන් සර් ගේ ඔලුවට රෙද්ද දාලා නෙල...,[],NOT,0
3,1397219745707986955,හොඳ මිතුරියක් කතා කලා. විස්තර කතාකරමින් ඉදලා ම...,හොඳ මිතුරියක් කතා කලා . විස්තර කතාකරමින් ඉදලා ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",OFF,1
4,950376113150222336,"ඔය බනින්නෙ.. හරකා, මී හරකා කිය කිය...","ඔය බනින්නෙ . . හරකා , මී හරකා කිය කිය . . .","[0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0]",OFF,1
...,...,...,...,...,...,...
7495,930270216612872193,අද උදේ දැක්කා පට්ට ලස්සන හීනයක්,අද උදේ දැක්කා පට්ට ලස්සන හීනයක්,[],NOT,0
7496,1159471424613969921,@USER කසල වෙන් කරලා දෙන්න පුරුදු වුනොත් වැඩේ ...,@USER කසල වෙන් කරලා දෙන්න පුරුදු වුනොත් වැඩේ ල...,[],NOT,0
7497,1073271775583100928,ඒත් පබාගේ පස්ස නම්... ඉස්සෝ කොටුවක් උනත් දාන්න...,ඒත් පබාගේ පස්ස නම් . . . ඉස්සෝ කොටුවක් උනත් දා...,"[0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...",OFF,1
7498,1131018244733657088,සුදුවෑන්වලට මාරම මාර බයක් තියෙන ඈයෝ අන්තවාදයට...,සුදුවෑන්වලට මාරම මාර බයක් තියෙන ඈයෝ අන්තවාදයට ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",OFF,1


In [4]:
 #=============================================================
# SECTION 2 — GENERAL TEXT CLEANER
# Works on BOTH Unicode Sinhala and Romanized Sinhala.
# Removes URLs, @mentions, hashtags, numbers, the literal
# string "URL", and excess whitespace.
# =============================================================
 
def clean_text(text):
    """
    General-purpose cleaner for social media Sinhala text.
    Safe to run on Unicode Sinhala — does not touch Sinhala chars.
    Safe to run on Romanized Sinhala — removes noise only.
 
    Removes:
      - http/https URLs
      - www. URLs
      - The literal token "URL" (appears in SOLD as a placeholder)
      - @mentions  → replaced with @USER for context
      - #hashtags
      - Standalone numbers (e.g. 1, 2024, 123)
      - Punctuation clutter (keeps Sinhala punctuation safe)
      - Extra whitespace
    """
    text = str(text).strip()
 
    # Remove http/https/www URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
 
    # Remove the literal placeholder token "URL" (uppercase, standalone)
    text = re.sub(r'\bURL\b', '', text)
 
    # # Anonymise @mentions
    # text = re.sub(r'@\w+', '@USER', text)
    #    # Anonymise @mentions
    # text = re.sub(r'@\w+', '@USER ', text)

    text = re.sub(r'@\w+', '', text)
  
 
    # Remove #hashtags entirely
    text = re.sub(r'#\w+', '', text)
 
    # Remove standalone numbers (not numbers attached to Sinhala words)
    text = re.sub(r'\b\d+\b', '', text)
 
    # Collapse multiple spaces / newlines into one space
    text = re.sub(r'\s+', ' ', text).strip()
 
    return text
 

In [5]:
# =============================================================
# SECTION 3 — CLEAN THE SOLD UNICODE DATA
# =============================================================
 
train_df["text_clean"] = train_df["text"].apply(clean_text)
test_df["text_clean"]  = test_df["text"].apply(clean_text)
 
# Verify cleaning — spot check
print("\n--- SOLD Unicode after cleaning (sample) ---")
print(train_df[["text", "text_clean", "label"]].head(5).to_string())
 


--- SOLD Unicode after cleaning (sample) ---
                                                                                                                                                                                                                                                                            text                                                                                                                                                                                                                                                                    text_clean label
0                                                                                                                                                                                                                                                     @USER @USER  පට්ට පට පට...                                                                                                                                     

In [6]:
test_df

,post_id,text,tokens,rationales,label,label_num,text_clean
0,995411387894542336,තේ නෙවෙයි තෝ බීලා ඉන්නෙ ගිනි වතුර,තේ නෙවෙයි තෝ බීලා ඉන්නෙ ගිනි වතුර,[],NOT,0,තේ නෙවෙයි තෝ බීලා ඉන්නෙ ගිනි වතුර
1,1128238706303864832,@USER තුනක් ඕනේ නෑ එකක් ගැහුවනම් ඇති ලංකාවට ආප...,@USER තුනක් ඕනේ නෑ එකක් ගැහුවනම් ඇති ලංකාවට ආප...,[],NOT,0,තුනක් ඕනේ නෑ එකක් ගැහුවනම් ඇති ලංකාවට ආපු ගමන්...
2,1290554471714496514,"@USER , @USER @USER ටහුඩු ඒකි තනියම ජීවත් වෙ...","@USER , @USER @USER ටහුඩු ඒකි තනියම ජීවත් වෙන්...",[],NOT,0,", ටහුඩු ඒකි තනියම ජීවත් වෙන්න වෙරදරන එක්කෙනෙක්..."
3,909459929446182913,@USER මර්විනුත් මුද්දරයක් වෙයිද දන්නේ නැ,@USER මර්විනුත් මුද්දරයක් වෙයිද දන්නේ නැ,[],NOT,0,මර්විනුත් මුද්දරයක් වෙයිද දන්නේ නැ
4,1305480579375198208,@USER ගොසිප් පීපල් නෙමේ හුත්තො. තෝ වගේ හොරෙක්ට...,@USER ගොසිප් පීපල් නෙමේ හුත්තො . තෝ වගේ හොරෙක්...,"[0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]",OFF,1,ගොසිප් පීපල් නෙමේ හුත්තො. තෝ වගේ හොරෙක්ට මේකෙ ...
...,...,...,...,...,...,...,...
2495,732381659518111744,"පට්ට වේසි, දැන් එනව මෙතන රහත් වෙන්න URL","පට්ට වේසි , දැන් එනව මෙතන රහත් වෙන්න URL","[1, 1, 0, 0, 0, 0, 0, 0, 0]",OFF,1,"පට්ට වේසි, දැන් එනව මෙතන රහත් වෙන්න"
2496,944199723564650496,"මේක හරි හුත්තක් උනානේ ""කොල්ලෙක් නෑ ,කෙල්ලෙක් ...","මේක හරි හුත්තක් උනානේ "" කොල්ලෙක් නෑ , කෙල්ලෙක්...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",OFF,1,"මේක හරි හුත්තක් උනානේ ""කොල්ලෙක් නෑ ,කෙල්ලෙක් න..."
2497,680949697666781184,ලොකු නංඟි ගෙ පරණ කොල්ලා ඒකිට වද දෙනවලු ඒ වුනාට...,ලොකු නංඟි ගෙ පරණ කොල්ලා ඒකිට වද දෙනවලු ඒ වුනාට...,[],NOT,0,ලොකු නංඟි ගෙ පරණ කොල්ලා ඒකිට වද දෙනවලු ඒ වුනාට...
2498,1184506690122633216,බය කරන්න එපා යකෝ සිත්තරා @USER · 16 Oct 2019...,බය කරන්න එපා යකෝ සිත්තරා @USER 16 Oct 2019 මේය...,[],NOT,0,බය කරන්න එපා යකෝ සිත්තරා · Oct මේය ලොව ඇති භයා...


In [7]:
# =============================================================
# SECTION 4 — LOAD AND CLEAN THE ROMANIZED EXCEL DATA
# =============================================================
 
romanized_raw = pd.read_excel("/kaggle/input/datasets/nethmishehara/romanized-data/Book1.xlsx")
 
# Your Excel has a header row buried as row 0 — fix that
# The real columns are: unicode_text, label, romanized_text, notes
romanized_raw.columns = ["unicode_text", "label", "romanized_text", "notes"]
romanized_raw = romanized_raw.iloc[1:].reset_index(drop=True)  # drop the fake header row
 
print(f"\nRomanized data loaded: {romanized_raw.shape}")
print(romanized_raw.head(3).to_string())
 
# Drop rows with missing romanized text or missing label
romanized_df = romanized_raw.dropna(subset=["romanized_text", "label"]).copy()
romanized_df = romanized_df[romanized_df["romanized_text"].str.strip() != ""].copy()
romanized_df = romanized_df.reset_index(drop=True)
 
# Map labels — handle both uppercase and lowercase just in case
romanized_df["label"]     = romanized_df["label"].str.strip().str.upper()
romanized_df["label_num"] = romanized_df["label"].map(label_map)
 
# Drop any rows where label didn't map (e.g. typos in your Excel)
bad_labels = romanized_df["label_num"].isna().sum()
if bad_labels > 0:
    print(f"\nWARNING: {bad_labels} rows had unrecognised labels and will be dropped.")
    print(romanized_df[romanized_df["label_num"].isna()]["label"].value_counts())
    romanized_df = romanized_df.dropna(subset=["label_num"]).reset_index(drop=True)
 
romanized_df["label_num"] = romanized_df["label_num"].astype(int)
 
print(f"\nRomanized data after cleaning: {romanized_df.shape}")
print(romanized_df["label"].value_counts())
 


Romanized data loaded: (300, 4)
                                                                  unicode_text label                                                                       romanized_text notes
0  “ඥානසාර හිමිගේ පිරිත් අසා මියපරලොව ගිය අගමැතිගේ අම්මාත් පුතාට සාප කරයි” URL   NOT  gnanasara himige pirith asa miyaparalowa giya agamathige ammath puthata saapa karai   NaN
1                                     DSLR එකක් නැති එකම DSLR පිස්සා මමද එතකොට   NOT                                     DSLR ekak nathi ekama DSLR pissa mamada ethakota   NaN
2                               @USER එල කිරි. මේ වගේ දෙයක් හරි වෙනවනම් හොදයි.   NOT                                                ela kiri me wage deyak hari wenawanam   NaN

Romanized data after cleaning: (300, 5)
label
NOT    150
OFF    150
Name: count, dtype: int64


In [8]:
import re
 
# =============================================================
# HATE SPEECH WORD DICTIONARY
# The ONLY words that get converted to Unicode.
# Everything else — Sinhala romanized, English, mixed — stays
# exactly as typed. This is intentional.
#
# Why this works:
# XLM-RoBERTa was pretrained on multilingual data including
# Sinhala Unicode. When it sees hate speech words in Unicode
# (හුත්තෝ, පාකයා, මෝඩයා) it recognises them from pretraining.
# When it sees them as Latin (hutto, pakaya, modaya) they are
# meaningless unknown tokens. Converting ONLY these words gives
# the model the signal it needs to classify hate speech correctly.
# =============================================================
 
HATE_WORDS = {
 
    # ── Direct insults and slurs ──────────────────────────────
    "huththo"       : "හුත්තෝ",
    "hutto"         : "හුත්තෝ",
    "hutho"         : "හුත්තෝ",
    "huththaa"      : "හුත්තා",
    "huththa"       : "හුත්ත",
    "pakaya"        : "පාකයා",
    "paka"          : "පාක",
    "pakayaa"       : "පාකයා",
    "modaya"        : "මෝඩයා",
    "moda"          : "මෝඩ",
    "modai"         : "මෝඩයි",
    "modayaa"       : "මෝඩයා",
    "pissu"         : "පිස්සු",
    "pissa"         : "පිස්සා",
    "pissuda"       : "පිස්සුද",
    "pissuwek"      : "පිස්සුවෙක්",
    "yakka"         : "යක්කා",
    "yako"          : "යකෝ",
    "balla"         : "බල්ලා",
    "ballaa"        : "බල්ලා",
    "ballanta"      : "බල්ලන්ට",
    "godaya"        : "ගොදයා",
    "godayaa"       : "ගොදයා",
    "kunu"          : "කුණු",
    "kunuharupa"    : "කුණුහරුප",
    "kunuharuwa"    : "කුණුහරුව",
    "baiya"         : "බෙයියා",
    "bayya"         : "බය්යා",
    "hora"          : "හොර",
    "horakam"       : "හොරකම්",
    "horayaa"       : "හොරයා",
    "kela"          : "කේළ",
    "kattiya"       : "කට්ටිය",
 
    # ── Targeting words (used when directing hate at someone) ─
    "umbata"        : "ඔඹට",
    "umba"          : "ඔඹ",
    "umbala"        : "ඔඹලා",
    "umbatama"      : "ඔඹටම",
 
    # ── Derogatory group labels ───────────────────────────────
    "demalu"        : "දෙමළු",
    "demalunta"     : "දෙමළුන්ට",
    "muslimayya"    : "මුස්ලිම්අය්යා",
    "kollantar"     : "කොල්ලන්තාර්",
 
    # ── Sexual / explicit hate terms ─────────────────────────
    "keli"          : "කෙළි",
    "wesige"        : "වේශිගේ",
    "vesi"          : "වේශී",
    "wesiya"        : "වේශියා",
    "hukana"        : "හූකන",
    "hukanawa"      : "හූකනවා",
 
    # ── Threatening / violent language ───────────────────────
    "maranawa"      : "මරනවා",
    "marapan"       : "මරාපන්",
    "gahanna"       : "ගහන්න",
    "kapanna"       : "කාපන්න",
    "nasanawa"      : "නසනවා",
 
    # ── Common hate speech qualifiers ────────────────────────
    "naraka"        : "නරක",
    "narakai"       : "නරකයි",
    "narak"         : "නරක",
    "apahu"         : "අපහු",
    "nidahas"       : "නිදහස්",
}
 
# Spelling variations of hate words → canonical form
# Checked BEFORE the main dictionary lookup
HATE_VARIATIONS = {
    # huththo variants
    "huthoo"    : "huththo",
    "hutho"     : "huththo",
    "huttho"    : "huththo",
    "huththa"   : "huththo",
    "huththoo"  : "huththo",
 
    # pakaya variants
    "pakayaa"   : "pakaya",
    "pakayyaa"  : "pakaya",
 
    # modaya variants
    "modayaa"   : "modaya",
    "moodaya"   : "modaya",
    "moodayaa"  : "modaya",
 
    # pissu variants
    "pissoo"    : "pissa",
    "pissaa"    : "pissa",
 
    # balla variants
    "ballaa"    : "balla",
    "balloo"    : "balla",
 
    # umba variants
    "umbata"    : "umbata",   # already in dict but alias for safety
    "umbataa"   : "umbata",
 
    # kunu variants
    "kunuu"     : "kunu",
    "kuunu"     : "kunu",
 
    # hora variants
    "horaa"     : "hora",
    "hoorana"   : "hora",
}
 
 
# =============================================================
# GENERAL TEXT CLEANER
# Removes URLs, @mentions, hashtags, the literal "URL" token,
# standalone numbers, and extra whitespace.
# Safe on both Unicode Sinhala and Romanized Sinhala.
# =============================================================
 
def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)   # real URLs
    text = re.sub(r'\bURL\b', '', text)                  # placeholder token
    text = re.sub(r'@\w+', '@USER', text)                # anonymise mentions
    text = re.sub(r'#\w+', '', text)                     # hashtags
    text = re.sub(r'\b\d+\b', '', text)                  # standalone numbers
    text = re.sub(r'\s+', ' ', text).strip()
    return text
 
 
# =============================================================
# NOISE CLEANER
# Fixes repeated characters common in social media typing.
# hondaaaa → honda, nehehehe → nehe
# =============================================================
 
def clean_noise(text):
    text = text.lower().strip()
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)   # keep max 2 repeats
    text = re.sub(r'\s+', ' ', text)
    return text
 
 
# =============================================================
# SCRIPT DETECTOR
# =============================================================
 
def is_romanized(text):
    """
    Returns True if less than 10% of non-space characters are
    Unicode Sinhala — meaning the text is Romanized Sinhala.
    """
    text = str(text)
    sinhala_chars = sum(1 for c in text if '\u0D80' <= c <= '\u0DFF')
    total = len(text.replace(' ', ''))
    if total == 0:
        return False
    return (sinhala_chars / total) < 0.1
 
 
# =============================================================
# CORE: TARGETED HATE WORD CONVERTER
# This is the main function. It scans a sentence word by word.
# If a word matches a hate speech term → convert to Unicode.
# Everything else → leave exactly as-is.
# =============================================================
 
def convert_hate_words(text):
    """
    Scan text word by word.
    Hate speech words → Unicode Sinhala.
    Everything else (Sinhala romanized, English, mixed) → unchanged.
 
    Input : "umbata huththo modaya this is bad"
    Output: "ඔඹට හුත්තෝ මෝඩයා this is bad"
    """
    words  = text.split()
    result = []
 
    for word in words:
        # Detach trailing punctuation so "hutto!" still matches "hutto"
        punct = ""
        if word and word[-1] in ".,!?;:\"'":
            punct = word[-1]
            word  = word[:-1]
 
        if not word:
            result.append(punct)
            continue
 
        w = word.lower().strip()
 
        # Step 1: check spelling variation → normalise to canonical
        canonical = HATE_VARIATIONS.get(w, w)
 
        # Step 2: check canonical form against hate word dictionary
        if canonical in HATE_WORDS:
            result.append(HATE_WORDS[canonical] + punct)
        else:
            # Not a hate word — keep exactly as typed
            result.append(word + punct)
 
    return " ".join(result)
 
 
# =============================================================
# FULL PREPROCESSING PIPELINE
# Entry point called from your training notebook.
# =============================================================
 
def preprocess(text):
    """
    Full preprocessing pipeline.
 
    Input  : Any Sinhala social media text (Unicode or Romanized)
    Output : (processed_text, script_type)
 
    Steps:
      1. General cleaning — URLs, @USER, URL token, numbers, hashtags
      2. Script detection — is it Romanized?
      3. If Romanized:
           a. Noise cleaning — fix hondaaaa → honda
           b. Hate word conversion — huththo → හුත්තෝ, umbata → ඔඹට
              (everything else stays as typed)
      4. If Unicode — return as-is after cleaning
    """
    text = str(text).strip()
 
    # Step 1: general cleaning (both script types)
    cleaned = clean_text(text)
 
    if is_romanized(cleaned):
        # Step 2a: fix repeated chars
        cleaned = clean_noise(cleaned)
        # Step 2b: convert only hate words to Unicode
        final = convert_hate_words(cleaned)
        return final, "romanized"
 
    return cleaned, "unicode"
 
 
# =============================================================
# QUICK TEST
# =============================================================
 
if __name__ == "__main__":
 
    test_cases = [
 
        # ── Pure hate speech — all should convert ─────────────
        ("umbata huththo modaya",
         "core hate words"),
 
        ("umbata pissuda yako pakaya",
         "multiple hate words"),
 
        ("oyaa naraka balla",
         "hate qualifiers + slur"),
 
        # ── Mixed English + Sinhala hate speech ───────────────
        ("you huththo crazy modaya",
         "English + Sinhala hate words"),
 
        ("this guy is a pakaya seriously",
         "English sentence with hate word"),
 
        ("umbata crazy bugger men yako",
         "your original failing test case"),
 
        # ── Spelling variants ─────────────────────────────────
        ("umbata huttho pisssaa modayaaa",
         "noise + variants"),
 
        ("huththooo ballaa kunuu",
         "repeated chars in hate words"),
 
        # ── Your Excel dataset rows ───────────────────────────
        ("gnanasara himige pirith asa miyaparalowa giya agamathige",
         "Excel row 0 — no hate words, stays as Latin"),
 
        ("ithin umba gona wage kata wahan hitiya man ahinsakayata",
         "Excel row 3 — umba converts, rest stays"),
 
        ("ketha wada ban yam rajathuma paduwe ekak dana minissunta ohoma balu wada karanna epa",
         "Excel row 7 — no hate words matched, stays as Latin"),
 
        ("kasada bandapu pirimiyek kiyala umbata onaawata hora",
         "Excel row 9 — umbata + hora convert"),
 
        # ── Unicode — should pass through unchanged ───────────
        ("@user කොහොමද ඔයා",
         "Unicode passthrough"),
 
        ("මේකා නරකයි URL https://t.co",
         "Unicode + URL removal"),
 
        ("@USER ගොසිප් නෙමේ හුත්තො. තෝ වගේ හොරෙක්ට",
         "Unicode hate speech — unchanged"),
    ]
 
    print("=" * 70)
    print("TARGETED HATE WORD CONVERTER — TEST RESULTS")
    print("Hate words → Unicode | Everything else → unchanged")
    print("=" * 70)
 
    for text, desc in test_cases:
        result, script = preprocess(text)
        changed = result != clean_text(text)
        print(f"\n[{desc}]")
        print(f"  Input  : {text}")
        print(f"  Script : {script}")
        print(f"  Output : {result}")
 
    print("\n" + "=" * 70)

TARGETED HATE WORD CONVERTER — TEST RESULTS
Hate words → Unicode | Everything else → unchanged

[core hate words]
  Input  : umbata huththo modaya
  Script : romanized
  Output : ඔඹට හුත්තෝ මෝඩයා

[multiple hate words]
  Input  : umbata pissuda yako pakaya
  Script : romanized
  Output : ඔඹට පිස්සුද යකෝ පාකයා

[hate qualifiers + slur]
  Input  : oyaa naraka balla
  Script : romanized
  Output : oyaa නරක බල්ලා

[English + Sinhala hate words]
  Input  : you huththo crazy modaya
  Script : romanized
  Output : you හුත්තෝ crazy මෝඩයා

[English sentence with hate word]
  Input  : this guy is a pakaya seriously
  Script : romanized
  Output : this guy is a පාකයා seriously

[your original failing test case]
  Input  : umbata crazy bugger men yako
  Script : romanized
  Output : ඔඹට crazy bugger men යකෝ

[noise + variants]
  Input  : umbata huttho pisssaa modayaaa
  Script : romanized
  Output : ඔඹට හුත්තෝ පිස්සා මෝඩයා

[repeated chars in hate words]
  Input  : huththooo ballaa kunuu
  Script 

In [9]:

# =============================================================
# SECTION 5 — APPLY TRANSLITERATION TO ROMANIZED DATA
# Paste your full transliteration_module.py above this section,
# or import it. The preprocess() function is called here.
# Output: Unicode Sinhala text ready for XLM-RoBERTa.
# =============================================================

# --- Paste or import your transliteration module here ---
# from transliteration_module import preprocess
# OR paste the full module code above this section in your notebook.

def apply_transliteration(df, text_col="romanized_text"):
    """
    Apply the full preprocess() pipeline to a romanized dataframe.
    Returns a new column 'text_clean' with Unicode Sinhala output.
    """
    results = []
    for idx, row in df.iterrows():
        raw = str(row[text_col]).strip()
        # Step 1: general cleaning first (URLs, numbers, hashtags)
        cleaned = clean_text(raw)
        # Step 2: transliteration pipeline
        unicode_out, script_type = preprocess(cleaned)
        results.append({"text_clean": unicode_out, "script_type": script_type})

    result_df = pd.DataFrame(results)
    df = df.copy()
    df["text_clean"]  = result_df["text_clean"].values
    df["script_type"] = result_df["script_type"].values
    return df

romanized_df = apply_transliteration(romanized_df, text_col="romanized_text")

# Verify transliteration output
print("\n--- Romanized data after transliteration (sample) ---")
print(romanized_df[["romanized_text", "text_clean", "label"]].head(10).to_string())

# Check script detection results
print("\nScript type distribution:")
print(romanized_df["script_type"].value_counts())


--- Romanized data after transliteration (sample) ---
                                                                                                                                                                                                                                romanized_text                                                                                                                                                                                                                               text_clean label
0                                                                                                                                                          gnanasara himige pirith asa miyaparalowa giya agamathige ammath puthata saapa karai                                                                                                                                                      gnanasara himige pirith asa miyaparalowa giya agamathige ammath puthata saapa k

In [10]:

# =============================================================
# SECTION 6 — BUILD EXPERIMENT DATASETS
# This is where you control which experiment you are running.
# =============================================================

#
# EXPERIMENT 1 — Unicode Baseline
# Only SOLD Unicode data, no romanized at all.
#
exp1_train = train_df[["text_clean", "label_num"]].copy()
exp1_test  = test_df[["text_clean", "label_num"]].copy()

# exp1_train = train_df[["text", "label_num"]].copy()
# exp1_test  = test_df[["text", "label_num"]].copy()

# exp1_train["text_clean"] = exp1_train["text"].apply(clean_text)
# exp1_test["text_clean"]  = exp1_test["text"].apply(clean_text)


#EXPERIMENT 2 — Romanized WITHOUT Transliteration
#Add romanized data but keep it as raw Latin text.
#Simulates what happens when XLM-RoBERTa sees unprocessed romanized.

romanized_no_translit = romanized_df[["romanized_text", "label_num"]].copy()
romanized_no_translit = romanized_no_translit.rename(columns={"romanized_text": "text_clean"})
romanized_no_translit["text_clean"] = romanized_no_translit["text_clean"].apply(clean_text)

exp2_train = pd.concat([train_df[["text_clean", "label_num"]], romanized_no_translit], ignore_index=True)
exp2_test  = test_df[["text_clean", "label_num"]].copy()

romanized_no_translit = romanized_df[["romanized_text", "label_num"]].copy()
romanized_no_translit = romanized_no_translit.rename(columns={"romanized_text": "text_clean"})
romanized_no_translit["text_clean"] = romanized_no_translit["text_clean"].apply(clean_text)

exp2_train = exp2_train = romanized_no_translit.copy()
exp2_test  = test_df[["text_clean", "label_num"]].copy()


#
# EXPERIMENT 3 — Romanized WITH Transliteration
# Add romanized data after full transliteration pipeline.
# This is what proves your pipeline works.
#
romanized_transliterated = romanized_df[["text_clean", "label_num"]].copy()

exp3_train = pd.concat([train_df[["text_clean", "label_num"]], romanized_transliterated], ignore_index=True)
exp3_test  = test_df[["text_clean", "label_num"]].copy()


#EXPERIMENT 4 — Mixed Training (Unicode + Romanized transliterated, shuffled)
#Usually the best performer. Same as Exp 3 but shuffled properly.

exp4_train = exp3_train.sample(frac=1, random_state=42).reset_index(drop=True)
exp4_test  = test_df[["text_clean", "label_num"]].copy()

# Print dataset sizes
for name, df_ in [("Exp1", exp1_train), ("Exp2", exp2_train),
                  ("Exp3", exp3_train), ("Exp4", exp4_train)]:
    print(f"{name} train size: {len(df_)} | OFF: {(df_['label_num']==1).sum()} | NOT: {(df_['label_num']==0).sum()}")



Exp1 train size: 7500 | OFF: 3176 | NOT: 4324
Exp2 train size: 300 | OFF: 150 | NOT: 150
Exp3 train size: 7800 | OFF: 3326 | NOT: 4474
Exp4 train size: 7800 | OFF: 3326 | NOT: 4474


In [11]:

# =============================================================
# SECTION 7 — TOKENIZER AND DATASET CLASS
# =============================================================

MODEL_NAME = "xlm-roberta-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class SinhalaDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts  = df["text_clean"].tolist()
        self.labels = df["label_num"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length     = self.max_len,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt",
        )
        return {
            "input_ids"      : encoding["input_ids"].squeeze(),
            "attention_mask" : encoding["attention_mask"].squeeze(),
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long),
        }



config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:

# =============================================================
# SECTION 8 — METRICS
# =============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float), dim=1).numpy()[:, 1]
    return {
        "accuracy" : accuracy_score(labels, preds),
        "macro_f1" : f1_score(labels, preds, average="macro"),
        "roc_auc"  : roc_auc_score(labels, probs),
    }



In [13]:

# =============================================================
# SECTION 9 — TRAINING FUNCTION
# Pass any experiment's train/test dataframe here.
# =============================================================

def run_experiment(exp_name, train_data, test_data):
    import shutil, os, gc

    print(f"\n{'='*60}")
    print(f"RUNNING: {exp_name}")
    print(f"Train size: {len(train_data)} | Test size: {len(test_data)}")
    print(f"{'='*60}")

    train_dataset = SinhalaDataset(train_data, tokenizer)
    test_dataset  = SinhalaDataset(test_data,  tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    # Use /tmp which has separate space from the Kaggle working dir
    tmp_dir = f"/tmp/exp_{exp_name.replace(' ', '_')}"

    training_args = TrainingArguments(
        output_dir                  = tmp_dir,
        num_train_epochs            = 3,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 32,
        warmup_steps                = 100,
        weight_decay                = 0.01,
        eval_strategy               = "epoch",
        save_strategy               = "no",       # NO checkpoint saving — saves ~3GB per experiment
        load_best_model_at_end      = False,       # must be False when save_strategy="no"
        metric_for_best_model       = "macro_f1",
        greater_is_better           = True,
        logging_dir                 = f"/tmp/logs_{exp_name.replace(' ', '_')}",
        logging_steps               = 50,
        fp16                        = True,
        report_to                   = "none",
    )

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = test_dataset,
        compute_metrics = compute_metrics,
    )

    trainer.train()

    # Final evaluation on last epoch weights
    results = trainer.evaluate()
    print(f"\n{exp_name} RESULTS:")
    print(f"  Accuracy  : {results['eval_accuracy']:.4f}")
    print(f"  Macro-F1  : {results['eval_macro_f1']:.4f}")
    print(f"  ROC-AUC   : {results['eval_roc_auc']:.4f}")

    # Save only Exp4 (the best/final model) to Kaggle output
    if exp_name == "Exp4_Mixed_Training":
        save_path = "/kaggle/working/final_model"
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  Final model saved to {save_path}")

    # Clean up /tmp to free space before next experiment
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir)

    # Free GPU and CPU memory
    del model, trainer, train_dataset, test_dataset
    gc.collect()
    torch.cuda.empty_cache()

    return results


In [14]:

# =============================================================
# SECTION 10 — RUN ALL 4 EXPERIMENTS
# Comment out the ones you don't want to run yet.
# =============================================================

results_summary = {}

# Experiment 1 — Unicode only baseline
r1 = run_experiment("Exp1_Unicode_Baseline", exp1_train, exp1_test)
results_summary["Exp1 Unicode Baseline"] = r1

# # Experiment 2 — Romanized without transliteration
# r2 = run_experiment("Exp2_Romanized_No_Translit", exp2_train, exp2_test)
# results_summary["Exp2 Romanized No Translit"] = r2

# # Experiment 3 — Romanized WITH transliteration (proves pipeline works)
# r3 = run_experiment("Exp3_Romanized_With_Translit", exp3_train, exp3_test)
# results_summary["Exp3 Romanized With Translit"] = r3

# Experiment 4 — Mixed training (best expected)
r4 = run_experiment("Exp4_Mixed_Training", exp4_train, exp4_test)
results_summary["Exp4 Mixed Training"] = r4


# =============================================================
# SECTION 11 — RESULTS TABLE
# Print a clean summary for your report.
# =============================================================

print("\n" + "="*65)
print("EXPERIMENT RESULTS SUMMARY")
print("="*65)
print(f"{'Experiment':<35} {'Accuracy':>10} {'Macro-F1':>10} {'ROC-AUC':>10}")
print("-"*65)
for name, r in results_summary.items():
    print(f"{name:<35} {r['eval_accuracy']:>10.4f} {r['eval_macro_f1']:>10.4f} {r['eval_roc_auc']:>10.4f}")
print("="*65)





RUNNING: Exp1_Unicode_Baseline
Train size: 7500 | Test size: 2500


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/l

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Roc Auc
1,1.095868,0.897151,0.809600,0.791746,0.880758
2,0.717748,0.802347,0.824800,0.810891,0.904221
3,0.668855,0.766470,0.833200,0.826451,0.915193


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Roc Auc
1,1.242753,1.532857,0.671600,0.668468,0.868154
2,0.867265,0.905628,0.810800,0.808811,0.905587
3,0.663883,0.788568,0.836400,0.830744,0.911139


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Exp4_Mixed_Training RESULTS:
  Accuracy  : 0.8364
  Macro-F1  : 0.8307
  ROC-AUC   : 0.9111


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Final model saved to /kaggle/working/final_model

EXPERIMENT RESULTS SUMMARY
Experiment                            Accuracy   Macro-F1    ROC-AUC
-----------------------------------------------------------------
Exp1 Unicode Baseline                   0.8332     0.8265     0.9152
Exp4 Mixed Training                     0.8364     0.8307     0.9111


In [15]:
romanized_only = romanized_df[["romanized_text", "label_num"]].copy()

print(romanized_only.head())
print(romanized_only["label_num"].value_counts())
print(romanized_only.head())
print(romanized_only["label_num"].value_counts())

                                      romanized_text  label_num
0  gnanasara himige pirith asa miyaparalowa giya ...          0
1   DSLR ekak nathi ekama DSLR pissa mamada ethakota          0
2              ela kiri me wage deyak hari wenawanam          0
3  ithin umba gona wage kata wahan hitiya man ahi...          1
4  curfew dammanam oya mona labbakwath naane yako...          0
label_num
0    150
1    150
Name: count, dtype: int64
                                      romanized_text  label_num
0  gnanasara himige pirith asa miyaparalowa giya ...          0
1   DSLR ekak nathi ekama DSLR pissa mamada ethakota          0
2              ela kiri me wage deyak hari wenawanam          0
3  ithin umba gona wage kata wahan hitiya man ahi...          1
4  curfew dammanam oya mona labbakwath naane yako...          0
label_num
0    150
1    150
Name: count, dtype: int64


In [16]:
from sklearn.model_selection import train_test_split

train_r, test_r = train_test_split(
    romanized_only,
    test_size=0.2,
    stratify=romanized_only["label_num"],
    random_state=42
)



In [17]:

# --- RAW Romanized ---
romanized_raw = romanized_df[["romanized_text", "label_num"]].copy()

romanized_raw = romanized_raw.rename(columns={
    "romanized_text": "text_clean"
})

romanized_raw["text_clean"] = romanized_raw["text_clean"].apply(clean_text)


In [18]:
# --- Transliteration version ---
romanized_translit = apply_transliteration(
    romanized_df.copy(),
    text_col="romanized_text"
)

romanized_translit = romanized_translit[["text_clean", "label_num"]].copy()

In [19]:
from sklearn.model_selection import train_test_split

# Create split using RAW (important for fairness)
train_idx, test_idx = train_test_split(
    romanized_raw.index,
    test_size=0.2,
    stratify=romanized_raw["label_num"],
    random_state=42
)

# EXP2 (RAW)
exp2_train = romanized_raw.loc[train_idx].reset_index(drop=True)
exp2_test  = romanized_raw.loc[test_idx].reset_index(drop=True)

# EXP3 (TRANSLIT — SAME SPLIT)
exp3_train = romanized_translit.loc[train_idx].reset_index(drop=True)
exp3_test  = romanized_translit.loc[test_idx].reset_index(drop=True)

In [20]:
print("EXP2 (RAW)")
print(exp2_train.head())

print("\nEXP3 (TRANSLIT)")
print(exp3_train.head())

print("\nLabel distribution:")
print(exp2_train["label_num"].value_counts())

EXP2 (RAW)
                                          text_clean  label_num
0  @USER '@USER me ballanta kiyanna deyakuth naeh...          1
1                  e unata eki innawanam thama patta          0
2  "bijja thiyagena aeta deken **nna laesthi wenn...          1
3                unath badala nathnam u kollek malli          0
4  ohep ... palayan ..... e kathawa mokakda .. ? ...          1

EXP3 (TRANSLIT)
                                          text_clean  label_num
0  @user '@user me බල්ලන්ට kiyanna deyakuth naeh ...          1
1                  e unata eki innawanam thama patta          0
2  "bijja thiyagena aeta deken **nna laesthi wenn...          1
3                unath badala nathnam u kollek malli          0
4  ohep .. palayan .. e kathawa mokakda .. ? e ki...          1

Label distribution:
label_num
1    120
0    120
Name: count, dtype: int64


In [21]:
exp2 = run_experiment("EXP2_Romanized_Raw",exp2_train, exp2_test, )

exp3 = run_experiment("EXP3_Romanized_Translit",exp3_train, exp3_test)


RUNNING: EXP2_Romanized_Raw
Train size: 240 | Test size: 60


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/l

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Roc Auc
1,No log,1.391506,0.500000,0.333333,0.623333
2,No log,1.389147,0.500000,0.333333,0.652222
3,No log,1.383689,0.500000,0.333333,0.671111


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/l

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Roc Auc
1,No log,1.391461,0.500000,0.333333,0.620000
2,No log,1.388908,0.500000,0.333333,0.662222
3,No log,1.376292,0.550000,0.455462,0.664444


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



EXP3_Romanized_Translit RESULTS:
  Accuracy  : 0.5500
  Macro-F1  : 0.4555
  ROC-AUC   : 0.6644
